In [ ]:
# pycryptodome ships its C backends (_raw_aes, _SHA256, ...) as raw shared libraries loaded via
# Crypto.Util._raw_api; importing them as modules always raises "does not define module export
# function", so that exact error is skipped. Crypto.SelfTest.__main__ is skipped because importing
# it runs the full self-test suite.
import importlib, pkgutil, warnings
import Crypto as _pkg_under_test

failed = []
for module_info in pkgutil.walk_packages(_pkg_under_test.__path__, prefix=f"{_pkg_under_test.__name__}."):
    if module_info.name.rsplit(".", 1)[-1] == "__main__":
        continue
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            importlib.import_module(module_info.name)
    except ImportError as e:
        if "does not define module export function" in str(e):
            continue  # raw shared library, not a Python module (see note above)
        failed.append((module_info.name, type(e).__name__, str(e)))
    except Exception as e:
        failed.append((module_info.name, type(e).__name__, str(e)))

if failed:
    for name, etype, msg in failed:
        print(f"FAIL {name}: {etype}: {msg}")
    raise AssertionError(f"{len(failed)} submodule(s) of {_pkg_under_test.__name__} failed to import")
print(f"OK — all submodules of {_pkg_under_test.__name__} imported cleanly (version {_pkg_under_test.__version__})")

In [ ]:
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes

key = get_random_bytes(32)
plaintext = b"Attack at dawn. " * 16
header = b"associated-data"

cipher = AES.new(key, AES.MODE_GCM)
cipher.update(header)
ciphertext, tag = cipher.encrypt_and_digest(plaintext)

dec = AES.new(key, AES.MODE_GCM, nonce=cipher.nonce)
dec.update(header)
assert dec.decrypt_and_verify(ciphertext, tag) == plaintext
print("AES-GCM round trip OK:", len(ciphertext), "byte ciphertext,", len(tag), "byte tag")

tampered = ciphertext[:-1] + bytes([ciphertext[-1] ^ 1])
bad = AES.new(key, AES.MODE_GCM, nonce=cipher.nonce)
bad.update(header)
try:
    bad.decrypt_and_verify(tampered, tag)
    raise AssertionError("tampered ciphertext must not verify")
except ValueError as e:
    print("tamper rejected:", e)

In [ ]:
from Crypto.Cipher import PKCS1_OAEP
from Crypto.PublicKey import RSA

key = RSA.generate(2048)
der = key.export_key(format="DER")
key2 = RSA.import_key(der)
assert key2.n == key.n and key2.d == key.d

ciphertext = PKCS1_OAEP.new(key.publickey()).encrypt(b"secret payload")
assert PKCS1_OAEP.new(key2).decrypt(ciphertext) == b"secret payload"
print("RSA OK:", key.size_in_bits(), "bits; DER round trip + OAEP decrypt verified")

In [ ]:
from Crypto.Hash import SHA256
from Crypto.PublicKey import ECC
from Crypto.Signature import DSS

key = ECC.generate(curve="p256")
signature = DSS.new(key, "fips-186-3").sign(SHA256.new(b"message to sign"))

verifier = DSS.new(key.public_key(), "fips-186-3")
verifier.verify(SHA256.new(b"message to sign"), signature)
print("ECC sign/verify OK on", key.curve)

try:
    verifier.verify(SHA256.new(b"tampered message"), signature)
    raise AssertionError("forged message must not verify")
except ValueError as e:
    print("forgery rejected:", e)

In [ ]:
import hashlib

from Crypto.Hash import BLAKE2b, SHA3_256, SHA256
from Crypto.Math.Numbers import Integer

data = b"The quick brown fox jumps over the lazy dog"
assert SHA256.new(data).hexdigest() == hashlib.sha256(data).hexdigest()
assert SHA3_256.new(data).hexdigest() == hashlib.sha3_256(data).hexdigest()
assert BLAKE2b.new(data=data, digest_bits=512).hexdigest() == hashlib.blake2b(data).hexdigest()
print("SHA-256 / SHA3-256 / BLAKE2b-512 all agree with hashlib")

# GMP present -> Crypto.Math._IntegerGMP; else bundled fallback. Either is valid.
print("Integer backend:", Integer.__module__)
assert (Integer(7).inverse(97) * 7) % 97 == 1
print("modular inverse invariant OK")

In [ ]:
from Crypto.Hash import SHA512
from Crypto.Protocol.KDF import PBKDF2, scrypt
from Crypto.Protocol.SecretSharing import Shamir
from Crypto.Random import get_random_bytes

salt = b"\x00" * 16
k1 = PBKDF2("correct horse", salt, dkLen=32, count=10000, hmac_hash_module=SHA512)
k2 = PBKDF2("correct horse", salt, dkLen=32, count=10000, hmac_hash_module=SHA512)
assert k1 == k2 and len(k1) == 32

s1 = scrypt("correct horse", salt, key_len=32, N=2**14, r=8, p=1)
s2 = scrypt("correct horse", salt, key_len=32, N=2**14, r=8, p=1)
assert s1 == s2 and s1 != k1
print("PBKDF2 + scrypt deterministic, 32-byte keys derived")

secret = get_random_bytes(16)
shares = Shamir.split(2, 3, secret)
assert Shamir.combine([shares[0], shares[2]]) == secret
print("Shamir 2-of-3: secret recovered from shares 1 and 3")